# 🎬 Dubly ME — Colab Pipeline Runner

This notebook bootstraps the **Dubly ME** video-dubbing pipeline on Google Colab.

**Architecture:**
- **Google Drive** → persistent storage for models, datasets, and outputs (survives disconnections)
- **Colab SSD (`/content/`)** → ephemeral compute workspace for code and **cached model weights** (fast I/O)
- **GitHub** → source of truth for code (public repo, no auth needed)

> ⚠️ **CRITICAL — FUSE Bottleneck Prevention:**
> Models are **copied from Google Drive → Colab local SSD** before loading.
> Never load model weights directly from `/content/drive/` — the FUSE mount has
> massive latency that will starve the GPU and cause timeouts.

---
## 0 — Runtime Verification
Verify that a GPU is allocated before we begin.

In [ ]:
# ── Verify GPU allocation ──────────────────────────────────────
import subprocess, sys

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    print("❌  No GPU detected. Go to Runtime → Change runtime type → GPU.")
    sys.exit(1)
else:
    # Print only the first 3 lines (GPU name + memory)
    for line in result.stdout.strip().split("\n")[:10]:
        print(line)
    print("\n✅  GPU runtime confirmed.")

---
## 1 — Mount Google Drive
Mount your Google Drive so we can access persistent model weights and datasets.

**First-time setup:** You will need to create the storage directory structure.
The cell below handles this automatically.

In [ ]:
# ── Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

print("✅  Google Drive mounted at /content/drive")

In [ ]:
# ── Create persistent directory structure (idempotent) ────────
import os

GDRIVE_BASE = "/content/drive/MyDrive/Dubly_ME_Storage"

GDRIVE_DIRS = [
    f"{GDRIVE_BASE}/models/whisper",
    f"{GDRIVE_BASE}/models/llm",
    f"{GDRIVE_BASE}/models/voice_cloning",
    f"{GDRIVE_BASE}/data/audio_in",
    f"{GDRIVE_BASE}/data/audio_out",
    f"{GDRIVE_BASE}/voice_profiles",
    f"{GDRIVE_BASE}/outputs/artifacts",
    f"{GDRIVE_BASE}/outputs/dubbed_segments",
]

for d in GDRIVE_DIRS:
    os.makedirs(d, exist_ok=True)
    print(f"  📁  {d}")

print(f"\n✅  Persistent storage ready at: {GDRIVE_BASE}")

---
## 2 — Clone / Update Code from GitHub
Clone the public repository into the Colab workspace.
If already cloned (from a previous cell execution), pull latest changes instead.

In [ ]:
# ── Clone or pull the repository ────────────────────────────────
import os

REPO_URL  = "https://github.com/Omar-Gemy/Dubly_ME.git"
REPO_DIR  = "/content/Dubly_ME"
BRANCH    = "feature/cloud-pipeline"  # ← change to 'main' after merge

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repository already cloned. Pulling latest from '{BRANCH}'…")
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    print(f"Cloning repository (branch: {BRANCH})…")
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}

print(f"\n✅  Code ready at: {REPO_DIR}")

---
## 3 — Install Dependencies
Install project-specific Python packages using the Colab-optimized requirements file.
This skips `torch`/`torchaudio` since they are pre-installed in the Colab runtime
and matched to the VM's CUDA version.

In [ ]:
# ── Install Colab-specific requirements ────────────────────────
!pip install -q -r /content/Dubly_ME/requirements-colab.txt

print("\n✅  Dependencies installed.")

---
## 4 — Copy Models: Google Drive → Colab Local SSD

> **🚨 CRITICAL — Why we do this:**
> Google Drive is mounted via FUSE, which has **massive I/O latency**.
> Loading a 3GB Whisper `large-v3` model directly from FUSE will starve the GPU
> and cause timeouts. We **must** copy model weights to Colab's local NVMe SSD
> (`/content/models/`) first, then load from there.
>
> This cell uses `rsync` for efficient delta-copying — if the models are already
> on the local SSD (e.g., you re-run this cell), only changed files are transferred.

In [ ]:
# ── Copy-then-Load: Move model weights to fast local storage ──
import os, shutil, time

# ── Path mapping: Google Drive (persistent) → Colab SSD (fast) ──
GDRIVE_MODELS = f"{GDRIVE_BASE}/models"
LOCAL_MODELS  = "/content/models"

os.makedirs(LOCAL_MODELS, exist_ok=True)

def copy_models_to_ssd(source_dir: str, dest_dir: str):
    """
    Recursively copy model directories from Google Drive to local SSD.
    Uses rsync for efficient delta transfers.
    Falls back to shutil.copytree if rsync is unavailable.
    """
    if not os.path.isdir(source_dir):
        print(f"  ⚠  Source not found: {source_dir}")
        print(f"      Upload your model weights to Google Drive first:")
        print(f"      {source_dir}")
        return

    # Check if there are any files to copy
    file_count = sum(len(files) for _, _, files in os.walk(source_dir))
    if file_count == 0:
        print(f"  ⚠  No model files found in: {source_dir}")
        print(f"      Upload your model weights to this Google Drive path first.")
        return

    print(f"  📦  Copying {file_count} file(s) from Google Drive → local SSD…")
    print(f"       Source:  {source_dir}")
    print(f"       Dest:    {dest_dir}")

    start = time.time()

    # Use rsync for efficient delta-copy (avoids re-copying unchanged files)
    rsync_result = os.system(f'rsync -ah --progress "{source_dir}/" "{dest_dir}/"')

    if rsync_result != 0:
        # Fallback to shutil if rsync fails
        print("  ⚠  rsync unavailable, falling back to shutil.copytree…")
        if os.path.isdir(dest_dir):
            shutil.rmtree(dest_dir)
        shutil.copytree(source_dir, dest_dir)

    elapsed = time.time() - start
    print(f"\n  ✅  Models copied in {elapsed:.1f}s")

# ── Execute the copy ──
copy_models_to_ssd(GDRIVE_MODELS, LOCAL_MODELS)

# ── Verify local model directory ──
print("\n📂  Local model cache contents:")
for root, dirs, files in os.walk(LOCAL_MODELS):
    level = root.replace(LOCAL_MODELS, "").count(os.sep)
    indent = "  " * (level + 1)
    print(f"{indent}📁 {os.path.basename(root)}/")
    sub_indent = "  " * (level + 2)
    for f in files:
        size_mb = os.path.getsize(os.path.join(root, f)) / (1024 * 1024)
        print(f"{sub_indent}📄 {f}  ({size_mb:.1f} MB)")

---
## 5 — Configure Runtime Paths
Set up all path variables that downstream pipeline stages will use.
These point to the **local SSD** for model loading (fast) and **Google Drive**
for persistent data I/O.

In [ ]:
# ── Runtime path configuration ─────────────────────────────────
import os

# Code location (cloned from GitHub)
REPO_DIR       = "/content/Dubly_ME"

# Fast local SSD — for model loading (avoids FUSE bottleneck)
LOCAL_MODELS   = "/content/models"

# Google Drive — persistent storage for data & outputs
GDRIVE_BASE    = "/content/drive/MyDrive/Dubly_ME_Storage"
GDRIVE_DATA    = f"{GDRIVE_BASE}/data"
GDRIVE_OUTPUTS = f"{GDRIVE_BASE}/outputs"

# ── Input files (read from Google Drive) ──
INPUT_AUDIO     = f"{GDRIVE_DATA}/audio_out/_temp_normalised.wav"
INPUT_SEGMENTS  = f"{GDRIVE_OUTPUTS}/artifacts/segments.json"

# ── Output files (written to Google Drive for persistence) ──
OUTPUT_TRANSCRIPTS = f"{GDRIVE_OUTPUTS}/artifacts/transcripts.json"

# ── Print summary ──
print("📋  Runtime Path Configuration:")
print(f"    Code repo:          {REPO_DIR}")
print(f"    Local model cache:  {LOCAL_MODELS}")
print(f"    GDrive storage:     {GDRIVE_BASE}")
print(f"    Input audio:        {INPUT_AUDIO}")
print(f"    Input segments:     {INPUT_SEGMENTS}")
print(f"    Output transcripts: {OUTPUT_TRANSCRIPTS}")
print("\n✅  Paths configured.")

---
## 6 — Run ASR Transcription Pipeline
Execute `asr_transcription.py` using its existing CLI interface.
All paths are passed as arguments — **no source code modifications needed**.

In [ ]:
# ── Run Phase C: ASR Transcription ────────────────────────────
#
# The script loads the Whisper model using the --model flag.
# faster-whisper auto-downloads models to its own cache, but if you
# have pre-downloaded weights in /content/models/whisper/, you can
# pass the directory path directly as the --model argument.
#
# All input/output paths are passed via CLI args so the script
# runs identically on Colab and locally — zero code changes.

# Option A: Use a model name (faster-whisper downloads & caches automatically)
MODEL_NAME = "large-v3-turbo"

# Option B: Use a pre-downloaded model directory from local SSD
# MODEL_NAME = "/content/models/whisper/large-v3-turbo"

!cd {REPO_DIR} && python src/asr_transcription.py \
    --model       {MODEL_NAME} \
    --device      cuda \
    --compute-type float16 \
    --source-lang auto \
    --input-audio     "{INPUT_AUDIO}" \
    --input-segments  "{INPUT_SEGMENTS}" \
    --output          "{OUTPUT_TRANSCRIPTS}"

---
## 7 — Verify Output
Quick sanity check: load the generated `transcripts.json` and display a summary.

In [ ]:
# ── Verify the output ──────────────────────────────────────────
import json

with open(OUTPUT_TRANSCRIPTS, "r", encoding="utf-8") as f:
    transcripts = json.load(f)

segments = transcripts.get("segments", [])
filled   = sum(1 for s in segments if s.get("text"))
total    = len(segments)

print(f"{'═' * 60}")
print(f"  📊  Transcription Results")
print(f"{'═' * 60}")
print(f"  Total segments:       {total}")
print(f"  With transcriptions:  {filled}")
print(f"  Empty/skipped:        {total - filled}")
print(f"  Source language:       {transcripts.get('source_language', 'N/A')}")
print(f"  ASR model:            {transcripts.get('asr_model', 'N/A')}")
print(f"  Completed at:         {transcripts.get('asr_completed_at', 'N/A')}")
print(f"{'═' * 60}")

# Preview first 5 transcribed segments
print("\n🔍  Preview (first 5 transcribed segments):")
count = 0
for seg in segments:
    text = seg.get("text", "").strip()
    if text:
        sid = seg.get("segment_id", "?")
        spk = seg.get("speaker", "?")
        preview = text[:80] + "…" if len(text) > 80 else text
        print(f"  [{sid}] Speaker {spk}: \"{preview}\"")
        count += 1
        if count >= 5:
            break

print(f"\n✅  Output saved to: {OUTPUT_TRANSCRIPTS}")
print("    (This file is on Google Drive — it persists after disconnection.)")